# CoPoLLM：SSH / GitHub 执行入口
首次手动连接 GPU，在此授权挂载 Drive。代码通过 GitHub 固定 commit 获取；SSH 用于查看日志、排错和同步。
以下默认只复核原有 10 条实验。先核查原始输出，再冻结验证集协议，最后开放全量评测。
完成、超时或失败后可自动释放整个运行时，SSH 也会断开。仅停止 Python 不会释放 GPU。
不要在 GitHub Actions 中自动抢占 Colab GPU。普通 Pro 不保证自动重新申请 GPU。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
assert Path('/content/drive/MyDrive').is_dir()


先在 SSH 中将 GitHub 仓库克隆到 `/content/llm-project`，切到本轮固定 commit；或修改下方 PROJECT。
GitHub 私有仓库使用你已经设置的 Git 认证，凭据不要写入 Notebook。安装后按提示重启会话，再运行后续格。


In [ ]:
import subprocess, sys
from pathlib import Path
PROJECT = Path('/content/llm-project')
assert (PROJECT/'reproduction/automation/evaluate.py').exists(), '请先克隆并同步项目'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PROJECT/'reproduction/automation/requirements-colab.txt')], check=True)


In [ ]:
from google.colab import drive
# 重启后重新执行挂载格，再执行本格。运行期间可通过 SSH 查看 Drive 日志。
import subprocess, sys, json
from pathlib import Path
PROJECT = Path('/content/llm-project')
OUT = Path('/content/drive/MyDrive/colab-llm-experiments/public_subset_eval')
STATE = OUT/'jobs'/'review_v1'
AUTO_RELEASE = True  # True 会在任务结束后释放整个 Colab 运行时并断开 SSH
BUDGET_MINUTES = 60
# 默认续跑现有 10 条，不能据此认为 100% 达到上限的基座输出已经修复。
command = [sys.executable, str(PROJECT/'reproduction/automation/job.py'),
           '--state-dir', str(STATE), '--timeout-minutes', str(BUDGET_MINUTES), '--',
           sys.executable, '-u', str(PROJECT/'reproduction/automation/evaluate.py'),
           '--out', str(OUT), '--run-name', 'qwen3_8b_nf4_ctx2048_v1', '--limit', '10']
try:
    result = subprocess.run(command, cwd=PROJECT)
    print('退出码：', result.returncode)
    if (STATE/'status.json').exists():
        print((STATE/'status.json').read_text())
finally:
    if AUTO_RELEASE:
        # 必须先确认 Drive 写回成功。若同步失败，本格会报错并保留运行时供人工处理。
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()


`job.py` 每几秒更新状态，日志追加到 STATE/job.log；同一个 STATE 只允许一个任务。
时间预算到达后终止任务，下一次从完整 JSONL 记录续跑；若断电留下损坏的末行会停止并要求核查，不静默丢数据。
强制终止、Notebook 进程崩溃或服务中断时 finally 不保证执行；此时用 Colab 菜单“断开并删除运行时”。
不要在 SSH 的独立 Python 进程中依赖 runtime.unassign()，它可能缺少 Notebook 消息通道。
